### 0. 환경 설정

- conda 가상환경\
conda create -n neo4j python=3.14\
conda activate neo4j\
pip install neo4j-graphrag neo4j openai

In [139]:
import os
from dotenv import load_dotenv

load_dotenv()

True

### 1. GraphDB 불러오기

- Neo4j Sandbox : https://sandbox.neo4j.com/

- sandbox 웹 > Movies 인스턴스 생성 > Connect via drivers (Python) > 연결 코드 copy \
 \
AttributeError: 'Session' object has no attribute 'read_transaction' \
-> Neo4j 드라이버 최신 버전(4.0->5.0)으로 해당 함수 더이상 지원 안함. \
-> session.execute_read() 함수로 대신 사용.



- Cypher 쿼리 \
favorite 제목 영화에 나온 배우들이 같이 나온 영화들

In [36]:
from neo4j import GraphDatabase, basic_auth

driver = GraphDatabase.driver(
  "bolt://44.204.35.129:7687",
  auth=basic_auth("neo4j", "piece-daughters-line"))

# cypher_query = '''
# MATCH (movie:Movie {title:$favorite})<-[:ACTED_IN]-(actor)-[:ACTED_IN]->(rec:Movie)
#  RETURN distinct rec.title as title LIMIT 20
# '''

# with driver.session(database="neo4j") as session:
#   results = session.execute_read(
#     lambda tx: tx.run(cypher_query,
#                       favorite="The Matrix").data())
#   for record in results:
#     print(record['title'])

# driver.close()


In [61]:
cypher_query = '''
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
WHERE m.title = 'Titanic' AND rec <> m
RETURN rec.title
LIMIT 20

'''

with driver.session(database="neo4j") as session:
  results = session.execute_read(
    lambda tx: tx.run(cypher_query).data())
  for record in results:
    print(record['title'])


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `IN_GENRE` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=19, offset=19>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 19, 'line': 2, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)\nWHERE m.title = 'Titanic' AND rec <> m\nRETURN rec.title\nLIMIT 20\n\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `I

### 2. Graph RAG 구현하기

- Text2Cypher Retriever로 Cypher 쿼리 생성
- 그래프 기반 검색

In [37]:
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.llm import OpenAILLM

# 쿼리텍스트를 기반으로 Cypher 쿼리문을 생성하고, Retrieval 후 답변을 생성할 때 사용할 LLM
llm = OpenAILLM(model_name="gpt-4o", model_params={"temperature": 0})

In [38]:
llm.invoke("What is the capital of France?")

LLMResponse(content='The capital of France is Paris.')

#### 2-1. Text2Cypher Retriever

https://github.com/neo4j/neo4j-graphrag-python/blob/main/src/neo4j_graphrag/retrievers/text2cypher.py

- Text2CypherRetriever 파라미터
    - neo4j db 드라이버
    - cypher 쿼리 생성하는 llm
    - Neo4j DB Schema
    - Input / Output Query 예시

- Neo4j DB Schema 형식

    ```
    Node properties:
    Person {name: STRING, born: INTEGER}
    Movie {tagline: STRING, title: STRING, released: INTEGER}
    Relationship properties:
    ACTED_IN {roles: LIST}
    REVIEWED {summary: STRING, rating: INTEGER}
    The relationships:
    (:Person)-[:ACTED_IN]->(:Movie)
    (:Person)-[:DIRECTED]->(:Movie)
    (:Person)-[:PRODUCED]->(:Movie)
    (:Person)-[:WROTE]->(:Movie)
    (:Person)-[:FOLLOWS]->(:Person)
    (:Person)-[:REVIEWED]->(:Movie)
    ```

In [39]:
from neo4j import GraphDatabase
from neo4j.time import Date

def get_node_datatype(value):
    """
        입력된 노드 Value의 데이터 타입을 반환하는 함수
    """
    if isinstance(value, str):
        return "STRING"
    elif isinstance(value, int):
        return "INTEGER"
    elif isinstance(value, float):
        return "FLOAT"
    elif isinstance(value, bool):
        return "BOOLEAN"
    elif isinstance(value, list):
        return f"LIST[{get_node_datatype(value[0])}]" if value else "LIST"
    elif isinstance(value, Date):
        return "DATE"
    else:
        return "UNKNOWN"

def get_schema(uri, user, password):
    """
        Graph DB의 정보를 받아 노드 및 관계의 프로퍼티를 추출하고 스키마 딕셔너리를 반환하는 함수
    """
    driver = GraphDatabase.driver(
        uri,
        auth=basic_auth(user, password))

    with driver.session() as session:
        # 노드 프로퍼티 및 타입 추출
        node_query = """
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        """
        nodes = session.run(node_query)

        # 관계 프로퍼티 및 타입 추출
        rel_query = """
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        """
        relationships = session.run(rel_query)

        # 관계 유형 및 방향 추출
        rel_direction_query = """
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        """
        rel_directions = session.run(rel_direction_query)

        # 스키마 딕셔너리 생성
        schema = {"nodes": {}, "relationships": {}, "relations": []}

        for record in nodes:
            label = record["label"]
            key = record["key"]
            sample_value = record["sample_value"] # 데이터 타입을 추론하기 위한 샘플 데이터
            inferred_type = get_node_datatype(sample_value)
            if label not in schema["nodes"]:
                schema["nodes"][label] = {}
            schema["nodes"][label][key] = inferred_type

        for record in relationships:
            rel_type = record["rel_type"]
            key = record["key"]
            sample_value = record["sample_value"] # 데이터 타입을 추론하기 위한 샘플 데이터
            inferred_type = get_node_datatype(sample_value)
            if rel_type not in schema["relationships"]:
                schema["relationships"][rel_type] = {}
            schema["relationships"][rel_type][key] = inferred_type

        for record in rel_directions:
            start_label = record["start_label"][0]
            rel_type = record["rel_type"]
            end_label = record["end_label"][0]
            schema["relations"].append(f"(:{start_label})-[:{rel_type}]->(:{end_label})")

        return schema

def format_schema(schema):
    """
        스키마 딕셔너리를 LLM에 제공하기 위해 원하는 형태로 formatting 하는 함수
    """
    result = []

    # 노드 프로퍼티 출력
    result.append("Node properties:")
    for label, properties in schema["nodes"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{label} {{{props}}}")

    # 관계 프로퍼티 출력
    result.append("Relationship properties:")
    for rel_type, properties in schema["relationships"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{rel_type} {{{props}}}")

    # 관계 프로퍼티 출력
    result.append("The relationships:")
    for relation in schema["relations"]:
        result.append(relation)

    return "\n".join(result)

In [40]:
# Neo4j DB Schema 제공
schema = get_schema("bolt://44.204.35.129:7687","neo4j", "piece-daughters-line")
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
Movie {tagline: STRING, title: STRING, released: INTEGER}
Person {name: STRING, born: INTEGER}
Relationship properties:
ACTED_IN {roles: LIST[STRING]}
REVIEWED {summary: STRING, rating: INTEGER}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:REVIEWED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)


- Retriever 예시 작성

    cursor한테 스키마 정보를 가지고 사용자 기반, 아이템 기반의 User INPUT과 Query문을 작성해달라고 함.

In [86]:
# LLM INPUT / QUERY 예시 제공
examples = [
# User-based 추천 예시
"USER INPUT: '저는 The Matrix를 5점으로 줬어요. 저와 비슷한 취향의 사람들이 높게 평가한 다른 영화를 추천해줘.' QUERY: MATCH (me:Person)-[r1:REVIEWED]->(m:Movie)<-[r2:REVIEWED]-(other:Person) WHERE m.title = 'The Matrix' AND r1.rating >= 4 AND r2.rating >= 4 AND me <> other MATCH (other)-[r3:REVIEWED]->(rec:Movie) WHERE r3.rating >= 4 AND NOT (me)-[:REVIEWED]->(rec) WITH rec, count(DISTINCT other) AS similar_users, avg(r3.rating) AS avg_rating RETURN rec.title AS title, similar_users, avg_rating ORDER BY similar_users DESC, avg_rating DESC LIMIT 10",

"USER INPUT: 'Tom Hanks와 취향이 비슷한 사람이 좋아한 영화를 알려줘.' QUERY: MATCH (me:Person {name: 'Tom Hanks'})-[r1:REVIEWED]->(m:Movie)<-[r2:REVIEWED]-(other:Person) WHERE r1.rating >= 4 AND r2.rating >= 4 AND me <> other MATCH (other)-[r3:REVIEWED]->(rec:Movie) WHERE r3.rating >= 4 AND NOT (me)-[:REVIEWED]->(rec) WITH rec, count(DISTINCT other) AS similar_users, avg(r3.rating) AS avg_rating RETURN rec.title AS title, similar_users, avg_rating ORDER BY similar_users DESC, avg_rating DESC LIMIT 10",

"""USER INPUT: 'Toy Story와 The Godfather 영화를 좋아하는 사람은 또 어떤 영화를 좋아하나요?' 
QUERY: // 1. 기준 영화 리스트를 변수로 처리 (확장성)
WITH ['Toy Story', 'The Godfather'] AS target_titles

// 2. 해당 영화들을 평점 4점 이상으로 준 사람 찾기
MATCH (p:Person)-[:REVIEWED]->(m:Movie)
WHERE m.title IN target_titles AND m.rating >= 4
WITH p, target_titles, count(DISTINCT m) AS num_liked
WHERE num_liked = size(target_titles) // 리스트 개수와 일치하는지 동적으로 체크

// 3. 그 사람들이 본 '다른' 영화들 탐색
MATCH (p)-[r:REVIEWED]->(rec:Movie)
WHERE NOT rec.title IN target_titles AND r.rating >= 4
RETURN rec.title AS title, 
       count(DISTINCT p) AS co_reviewers, 
       avg(r.rating) AS avg_rating
ORDER BY co_reviewers DESC, avg_rating DESC
LIMIT 10
""",
# Item-based 추천 예시
"USER INPUT: 'The Matrix와 비슷한 영화를 추천해줘.' QUERY: MATCH (target:Movie {title: 'The Matrix'})<-[:ACTED_IN]-(a:Person)-[:ACTED_IN]->(rec:Movie) WHERE rec <> target WITH target, rec, count(DISTINCT a) AS common_actors OPTIONAL MATCH (target)<-[:DIRECTED]-(d:Person)-[:DIRECTED]->(rec) WITH target, rec, common_actors, count(DISTINCT d) AS common_directors OPTIONAL MATCH (target)<-[:WROTE]-(w:Person)-[:WROTE]->(rec) WITH rec, common_actors, common_directors, count(DISTINCT w) AS common_writers RETURN rec.title AS title, common_actors, common_directors, common_writers, (3*common_actors + 4*common_directors + 2*common_writers) AS score ORDER BY score DESC, title ASC LIMIT 10",

"USER INPUT: 'Apollo 13를 좋아한 사람들이 같이 높게 평가한 영화를 추천해줘.' QUERY: MATCH (target:Movie {title: 'Apollo 13'})<-[rt:REVIEWED]-(u:Person)-[rr:REVIEWED]->(rec:Movie) WHERE rec <> target AND rt.rating >= 4 AND rr.rating >= 4 WITH rec, count(DISTINCT u) AS co_reviewers, avg(rr.rating) AS avg_rating RETURN rec.title AS title, co_reviewers, avg_rating ORDER BY co_reviewers DESC, avg_rating DESC LIMIT 10"
]

In [87]:
# Text2CypherRetriever
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,  # type: ignore
    neo4j_schema=neo4j_schema,
    examples=examples,
)

# LLM을 통해 Cypher 쿼리를 생성하여 Neo4j DB에 보내고, 그 결과를 반환 => 이 결과는 RAG에 활용됨
query_text = "Tom Hanks 가 어떤 영화에 출연했나요?"
search_result = retriever.search(query_text=query_text)
search_result

RetrieverResult(items=[RetrieverResultItem(content="<Record title='Apollo 13'>", metadata=None), RetrieverResultItem(content='<Record title="You\'ve Got Mail">', metadata=None), RetrieverResultItem(content="<Record title='A League of Their Own'>", metadata=None), RetrieverResultItem(content="<Record title='Joe Versus the Volcano'>", metadata=None), RetrieverResultItem(content="<Record title='That Thing You Do'>", metadata=None), RetrieverResultItem(content="<Record title='The Da Vinci Code'>", metadata=None), RetrieverResultItem(content="<Record title='Cloud Atlas'>", metadata=None), RetrieverResultItem(content="<Record title='Cast Away'>", metadata=None), RetrieverResultItem(content="<Record title='The Green Mile'>", metadata=None), RetrieverResultItem(content="<Record title='Sleepless in Seattle'>", metadata=None), RetrieverResultItem(content="<Record title='The Polar Express'>", metadata=None), RetrieverResultItem(content='<Record title="Charlie Wilson\'s War">', metadata=None)], me

In [64]:
search_result.items

[RetrieverResultItem(content="<Record title='Joe Versus the Volcano'>", metadata=None),
 RetrieverResultItem(content="<Record title='A League of Their Own'>", metadata=None),
 RetrieverResultItem(content="<Record title='Sleepless in Seattle'>", metadata=None),
 RetrieverResultItem(content="<Record title='Apollo 13'>", metadata=None),
 RetrieverResultItem(content="<Record title='That Thing You Do'>", metadata=None),
 RetrieverResultItem(content='<Record title="You\'ve Got Mail">', metadata=None),
 RetrieverResultItem(content="<Record title='The Green Mile'>", metadata=None),
 RetrieverResultItem(content="<Record title='Cast Away'>", metadata=None),
 RetrieverResultItem(content="<Record title='The Polar Express'>", metadata=None),
 RetrieverResultItem(content="<Record title='The Da Vinci Code'>", metadata=None),
 RetrieverResultItem(content='<Record title="Charlie Wilson\'s War">', metadata=None),
 RetrieverResultItem(content="<Record title='Cloud Atlas'>", metadata=None)]

In [65]:
search_result.metadata['cypher']

"MATCH (p:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie) RETURN m.title AS title ORDER BY m.released ASC"

In [67]:
query_text = "저는 Joe Versus the Volcano을 좋아합니다. 비슷한 장르의 영화를 추천해줄 수 있나요?"
search_result = retriever.search(query_text=query_text)
print("cypher 쿼리--------")
print(search_result.metadata['cypher'])
print("결과---------")
for item in search_result.items:
    print(item)

cypher 쿼리--------
MATCH (target:Movie {title: 'Joe Versus the Volcano'})<-[:ACTED_IN]-(a:Person)-[:ACTED_IN]->(rec:Movie) 
WHERE rec <> target 
WITH target, rec, count(DISTINCT a) AS common_actors 
OPTIONAL MATCH (target)<-[:DIRECTED]-(d:Person)-[:DIRECTED]->(rec) 
WITH target, rec, common_actors, count(DISTINCT d) AS common_directors 
OPTIONAL MATCH (target)<-[:WROTE]-(w:Person)-[:WROTE]->(rec) 
WITH rec, common_actors, common_directors, count(DISTINCT w) AS common_writers 
RETURN rec.title AS title, common_actors, common_directors, common_writers, (3*common_actors + 4*common_directors + 2*common_writers) AS score 
ORDER BY score DESC, title ASC 
LIMIT 10
결과---------
content="<Record title='Sleepless in Seattle' common_actors=2 common_directors=0 common_writers=0 score=6>" metadata=None
content='<Record title="You\'ve Got Mail" common_actors=2 common_directors=0 common_writers=0 score=6>' metadata=None
content="<Record title='A League of Their Own' common_actors=1 common_directors=0 c

#### 2.2 RAG 생성

https://github.com/neo4j/neo4j-graphrag-python/blob/main/src/neo4j_graphrag/generation/graphrag.py

In [88]:
from neo4j_graphrag.generation import GraphRAG
# RAG 파이프라인 초기화
rag = GraphRAG(retriever=retriever, llm=llm)

In [81]:
query_text="Titanic 과 비슷한 장르의 영화 추천해주세요"

response = rag.search(query_text, return_context=True)
print("==== [Text2Cypher 를 통해 자동생성한 Cypher] ====")
print(response.retriever_result.metadata['cypher'])
print("\n==== [생성된 Cypher를 기반으로 최종답변생성] ====")
print(response.answer)

==== [Text2Cypher 를 통해 자동생성한 Cypher] ====
MATCH (target:Movie {title: 'Titanic'})<-[:ACTED_IN]-(a:Person)-[:ACTED_IN]->(rec:Movie) WHERE rec <> target WITH target, rec, count(DISTINCT a) AS common_actors OPTIONAL MATCH (target)<-[:DIRECTED]-(d:Person)-[:DIRECTED]->(rec) WITH target, rec, common_actors, count(DISTINCT d) AS common_directors OPTIONAL MATCH (target)<-[:WROTE]-(w:Person)-[:WROTE]->(rec) WITH rec, common_actors, common_directors, count(DISTINCT w) AS common_writers RETURN rec.title AS title, common_actors, common_directors, common_writers, (3*common_actors + 4*common_directors + 2*common_writers) AS score ORDER BY score DESC, title ASC LIMIT 10

==== [생성된 Cypher를 기반으로 최종답변생성] ====
"Titanic"과 비슷한 장르의 영화로는 "The Notebook", "Atonement", "Pride and Prejudice", "The Great Gatsby", "Romeo + Juliet", "The English Patient", "Out of Africa", "Cold Mountain", "The Age of Innocence", 그리고 "The Painted Veil" 등을 추천드립니다. 이 영화들은 로맨스와 드라마 요소를 결합하여 감동적인 이야기를 전달합니다.


In [89]:
query_text = "Toy Story와 The Godfather 영화를 좋아하는 사람은 또 어떤 영화를 좋아하나요?"

response = rag.search(query_text, return_context=True)
print("==== [Text2Cypher 를 통해 자동생성한 Cypher] ====")
print(response.retriever_result.metadata['cypher'])
print("\n==== [생성된 Cypher를 기반으로 최종답변생성] ====")
print(response.answer)

==== [Text2Cypher 를 통해 자동생성한 Cypher] ====
cypher
WITH ['Toy Story', 'The Godfather'] AS target_titles
MATCH (p:Person)-[r:REVIEWED]->(m:Movie)
WHERE m.title IN target_titles AND r.rating >= 4
WITH p, target_titles, count(DISTINCT m) AS num_liked
WHERE num_liked = size(target_titles)
MATCH (p)-[r2:REVIEWED]->(rec:Movie)
WHERE NOT rec.title IN target_titles AND r2.rating >= 4
RETURN rec.title AS title, 
       count(DISTINCT p) AS co_reviewers, 
       avg(r2.rating) AS avg_rating
ORDER BY co_reviewers DESC, avg_rating DESC
LIMIT 10


==== [생성된 Cypher를 기반으로 최종답변생성] ====
Toy Story와 The Godfather 영화를 좋아하는 사람은 다양한 장르의 영화를 즐길 가능성이 높습니다. 예를 들어, Toy Story와 같은 애니메이션이나 가족 영화로는 "Finding Nemo"나 "Shrek"을, The Godfather와 같은 범죄 드라마로는 "Goodfellas"나 "Scarface"를 좋아할 수 있습니다. 또한, 두 영화 모두 스토리텔링이 뛰어난 작품이기 때문에 "The Shawshank Redemption"이나 "Pulp Fiction"과 같은 영화도 선호할 수 있습니다.


### 3. Gradio로 챗봇 배포하기

In [ ]:
import gradio as gr
from gradio.themes.base import Base


c:\Users\dyyu\anaconda3\envs\neo4j\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [117]:
## gradio 챗봇 기본 구조

# gradio library를 가져온다. gr로 줄여서 칭한다.
import gradio as gr

# gradio 내부에서 사용할 함수다.
def update(name): 
    return f"Welcome to Gradio, {name}!"

with gr.Blocks() as iface:
    gr.Markdown("Hello, World! I’m yelling in gradio!")
    with gr.Row():
        inp = gr.Textbox(placeholder="이름이 무엇인가요?")
        out = gr.Textbox()
    btn = gr.Button("제출")

    # 버튼에 이벤트 리스너를 추가한다. 
    # 버튼 클릭시 update함수를 호출하고, inp에 입력된 문자열을 파라미터로 보낸다. 함수의 반환값은 out에 출력한다.
    btn.click(fn=update, inputs=inp, outputs=out)

iface.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [137]:
import gradio as gr
import re
from typing import Any, List

def extract_title_from_item(item: Any) -> str:
    content = getattr(item, "content", item)
    if not isinstance(content, str):
        content = str(content)

    m = re.search(r"title='([^']+)'", content)
    if m:
        return m.group(1)

    return content

def format_items(items: List[Any], limit: int = 10) -> str:
    titles = [extract_title_from_item(it) for it in (items or [])][:limit]
    if not titles:
        return "결과가 없습니다."
    return "\n".join([f"- {t}" for t in titles])

# LLM에게 질문의 주제를 판단하게 하는 함수 (간단한 예시)
def is_movie_query(message: str) -> bool:
    # # 키워드 기반 필터링 (가장 빠르지만 정확도는 낮음)
    # movie_keywords = ["영화", "배우", "감독", "출연", "추천", "평점", "개봉", "무비", "연기"]
    # if any(kw in message for kw in movie_keywords):
    #     return True
    
    # 혹은 LLM을 사용하여 판단 (권장)
    prompt = f"사용자 질문: '{message}'\n이 질문이 영화, 배우, 감독 등 영화 관련 주제인가요? 오직 'True' 또는 'False'로만 답하세요."
    response = llm.invoke(prompt)
    # response가 객체인 경우 내용을 추출 (LangChain 기준)
    content = response.content.strip().lower() if hasattr(response, 'content') else str(response).strip().lower()
    # "true"라는 단어가 포함되어 있는지 확인
    return "true" in content
    # return True # 일단 기본적으로 True 반환 (로직 연결용)

def chat_fn(message: str, history):


    if not is_movie_query(message):
        return "죄송합니다. 저는 영화와 관련된 정보만 제공할 수 있는 챗봇입니다. 영화 제목이나 배우, 감독에 대해 물어봐 주세요! 🎬"

    try:
        rag_result = rag.search(
            query_text=message,
            return_context=True
        )

        answer = rag_result.answer
        cypher = rag_result.retriever_result.metadata.get("cypher", "")
        items = rag_result.retriever_result.items

        evidence = (
            "## 추천 후보\n"
            + format_items(items, limit=10)
            + "\n\n---\n## 실행된 Cypher\n"
            + "```sql\n"
            + (cypher or "(cypher 없음)")
            + "\n```"
        )
        return answer + "\n\n" + evidence

    except Exception as e:
        return f"오류가 발생했어요: {e}"

demo = gr.ChatInterface(
    fn=chat_fn,
    title="GraphRAG Movie Chatbot",
    description="chatbot 설명: Neo4j 그래프 기반 영화 추천 (실행 Cypher 쿼리 포함)"
)

demo.launch(debug=True, share=False)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
